
#### Run the cell below to install the required packages for Copilot


In [ ]:
!pip install azure-ai-projects

In [ ]:
PROJECT_CONNECTION_STRING=""
TENANT_ID=""
CLIENT_ID=""
CLIENT_SECRET=""
AI_SKILL_ENDPOINT=""

In [ ]:
# Define custom functions

from synapse.ml.mlflow import get_mlflow_env_config
import requests
import json


def ask_about_data(question: str) -> str:
    """
    Sends a natural language query to the holiday database and returns the result. Use this to ask about the following: Public holidays.

    :param question (str): The natural language query to be sent. Should be equivalent to one SQL query.
    :return: Result of the query.
    :rtype: str
    """

    print(f"Calling AI Skill with question: {question}")

    configs = get_mlflow_env_config()
    token = configs.driver_aad_token

    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json; charset=utf-8"
    }

    query = f'{{userQuestion:"{question}"}}'

    response = requests.post(AI_SKILL_ENDPOINT, headers=headers, data = query)
    response_json = json.loads(response.content)

    return response_json["result"]

In [ ]:
import os
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import FunctionTool, ToolSet, CodeInterpreterTool
from azure.identity import ClientSecretCredential
from typing import Any
from pathlib import Path

# Create an Azure AI Client from a connection string, copied from your Azure AI Foundry project.
# At the moment, it should be in the format "<HostName>;<AzureSubscriptionId>;<ResourceGroup>;<ProjectName>"
# HostName can be found by navigating to your discovery_url and removing the leading "https://" and trailing "/discovery"
# To find your discovery_url, run the CLI command: az ml workspace show -n {project_name} --resource-group {resource_group_name} --query discovery_url
# Project Connection example: eastus.api.azureml.ms;12345678-abcd-1234-9fc6-62780b3d3e05;my-resource-group;my-project-name
# Customer needs to login to Azure subscription via Azure CLI and set the environment variables

# Create an app registration with permissions on the Azure AI Foundry project. Note supported regions for Azure AI Agent Service (e.g., East US).

project_client = AIProjectClient.from_connection_string(
    credential=ClientSecretCredential(
        tenant_id=TENANT_ID,
        client_id=CLIENT_ID,
        client_secret=CLIENT_SECRET
    ), conn_str=PROJECT_CONNECTION_STRING
)

with project_client:
    # Create an instance of the CodeInterpreterTool
    code_interpreter = CodeInterpreterTool()

    # Initialize agent toolset with user functions
    functions = FunctionTool([ask_about_data])
    toolset = ToolSet()
    toolset.add(code_interpreter)
    toolset.add(functions)

    # The CodeInterpreterTool needs to be included in creation of the agent
    agent = project_client.agents.create_agent(
        model="gpt-4o-mini",
        name="my-agent",
        instructions="You are helpful agent. You have tools to help you with specialized knowledge to inform your responses. Always use these tools when applicable.",
        toolset=toolset,
        tool_resources=code_interpreter.resources,
    )
    print(f"Created agent, agent ID: {agent.id}")

    # Create a thread
    thread = project_client.agents.create_thread()
    print(f"Created thread, thread ID: {thread.id}")

    # Create a message
    message = project_client.agents.create_message(
        thread_id=thread.id,
        role="user",
        content="Compare number of public holidays in Denmark in 2023-2024. Create a bar chart based on that data.",
    )
    print(f"Created message, message ID: {message.id}")

    # Run the agent
    run = project_client.agents.create_and_process_run(thread_id=thread.id, assistant_id=agent.id)
    print(f"Run finished with status: {run.status}")

    if run.status == "failed":
        # Check if you got "Rate limit is exceeded.", then you want to get more quota
        print(f"Run failed: {run.last_error}")

    # Get messages from the thread
    messages = project_client.agents.list_messages(thread_id=thread.id)
    print(f"Messages: {messages}")

    # Get the last message from the sender
    last_msg = messages.get_last_text_message_by_role("assistant")
    if last_msg:
        print(f"Last Message: {last_msg.text.value}")

    # Print the file path(s) from the messages
    paths = []
    for image_content in messages.image_contents:
        file_id = image_content.image_file.file_id
        print(f"Image File ID: {file_id}")
        file_name = f"{file_id}_image_file.png"
        project_client.agents.save_file(file_id=file_id, file_name=file_name)
        print(f"Saved image file to: {Path.cwd() / file_name}")
        paths.append(f"{Path.cwd() / file_name}")

    for file_path_annotation in messages.file_path_annotations:
        print(f"File Paths:")
        print(f"Type: {file_path_annotation.type}")
        print(f"Text: {file_path_annotation.text}")
        print(f"File ID: {file_path_annotation.file_path.file_id}")
        print(f"Start Index: {file_path_annotation.start_index}")
        print(f"End Index: {file_path_annotation.end_index}")
        project_client.agents.save_file(file_id=file_path_annotation.file_path.file_id, file_name=Path(file_path_annotation.text).name)
        paths.append(Path(file_path_annotation.text).name)

    # Delete the agent once done
    project_client.agents.delete_agent(agent.id)
    print("\nDeleted agent")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

for path in paths:
    # Load the image
    img = mpimg.imread(path)

    # Display the image
    plt.imshow(img)
    plt.axis('off')  # Hide the axis
    plt.show()